# Trivial baselines on track 1: majority and seeded random

The floor every real system has to clear, scored with the official scorer on the dev set
(our course test set). Two systems:

- **majority**: the most frequent training label. Track-1 train ties FORWARD_ENTAILMENT and
  BACKWARD_ENTAILMENT at 881 each; `majority_label` breaks ties by canonical label order, so
  the prediction is FORWARD_ENTAILMENT everywhere.
- **random (seed 13)**: one uniform label per instance from `random.Random(13)`.

Everything this notebook reports lands in `experiments/<RUN_DATE>-trivial-baselines-track<TRACK>/`
(`config.json`, `predictions/`, the scorer's own `scores/<system>/`, `results.md`).

Run headlessly from the repository root with
`PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/01_trivial_baselines.ipynb`.
Needs the task repository cloned at `datasets/dico-nli/` (see `datasets/dico-nli.json`).

In [ ]:
from datetime import date

import pandas as pd

from dico_nli.baselines.trivial import majority_label, predict_majority, predict_random
from dico_nli.data import DATA_ROOT, load_reference
from dico_nli.experiments import experiment_dir, write_config, write_results_table
from dico_nli.scoring import score, write_predictions

TRACK = 1
SEED = 13
RUN_DATE = date(2026, 9, 29)   # the folder this run is recorded under; reruns replicate into it
OVERWRITE = True               # a rerun is deliberate here; the guard exists for scripts that might loop

## Data

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
gold = DATA_ROOT / "dev" / f"dico_nli_dev_track{TRACK}_reference.csv"
out = experiment_dir(f"trivial-baselines-track{TRACK}", day=RUN_DATE, overwrite=OVERWRITE)
print(f"train {len(train)}  dev {len(dev)}  majority label in train: {majority_label(train)}")

## Predict and score

Predictions are written in the official two-column format and scored by the official scorer, run as a subprocess exactly as CodaBench runs it.

In [ ]:
systems = {
    "majority": predict_majority(train, dev),
    f"random (seed {SEED})": predict_random(dev, SEED),
}
rows = {}
for name, predictions in systems.items():
    key = name.split(" ")[0]
    prediction_path = write_predictions(out / "predictions" / f"{key}.csv", predictions)
    rows[name] = score(gold, prediction_path, out / "scores" / key)

pd.DataFrame({name: {"weighted_f1": s.weighted_f1, "soft_cons": s.soft_cons, "hard_cons": s.hard_cons}
              for name, s in rows.items()}).T.round(4)

## Run record

Seed, data commit, repository commit (and whether the tree was dirty) and the exact command, next to the results table.

In [ ]:
write_config(
    out, seed=SEED, track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/01_trivial_baselines.ipynb",
    extra={"majority_label": majority_label(train), "train_size": len(train), "dev_size": len(dev), "systems": list(systems)},
)
print(write_results_table(out, rows, title=f"Trivial baselines, track {TRACK} dev").read_text(encoding="utf-8"))